# MobileNetV2 en Colab — PlantVillage (autocontenido)

Notebook autocontenido para Google Colab: todo el código vive en las celdas; la única descarga externa es el dataset PlantVillage. No importa del paquete `leafthealthvision` ni de archivos locales del proyecto.

Ruta F: comparación con EfficientNetB0 bajo mismas particiones, resolución, preprocesamiento y métricas.

> **Advertencia:** resultado orientativo, no diagnóstico agrícola. No declara cumplimiento completo del proyecto ni generalización a campo.


In [ ]:
# 0. Instalar dependencias (solo librerías externas; el código del proyecto va en las celdas)
!pip -q install tensorflow==2.21.0 mlflow==3.18.0 codecarbon>=3.3.2 scikit-learn==1.7.2 matplotlib==3.10.9


In [ ]:
# 1. Descargar el dataset real de PlantVillage (única descarga externa de datos)
import os, shutil, subprocess
from pathlib import Path

WORK = Path("/content/leafthealthvision")
DATA_DIR = WORK / "data" / "plantvillage"
REPO_URL = "https://github.com/spMohanty/PlantVillage-Dataset.git"

if not DATA_DIR.exists():
    WORK.mkdir(parents=True, exist_ok=True)
    print(f"Clonando {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(DATA_DIR)], check=True)
    git_dir = DATA_DIR / ".git"
    if git_dir.exists():
        shutil.rmtree(git_dir)
    for item in DATA_DIR.iterdir():
        if item.name not in {"raw", "leaf_grouping", "leaf-map.json", "README.md", "CITATION.cff"}:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
    print("Dataset listo en:", DATA_DIR)
else:
    print("Dataset ya descargado en:", DATA_DIR)

data_dir = DATA_DIR / "raw" / "color"
print("Imágenes en:", data_dir)
print("Clases encontradas:", len([p for p in data_dir.iterdir() if p.is_dir()]))


In [ ]:
# 2. Configuración (equivalente a src/leafthealthvision/config/settings.py)
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU activada: {len(gpus)} dispositivo(s)")
    except RuntimeError as e:
        print(f"Nota GPU (ya inicializada): {e}")
else:
    print("No se detectó GPU; se usará CPU")

SEED = 42
BATCH_SIZE = 32
IMAGE_TARGET_SIZE = (224, 224)
OPTIMIZER = "adam"
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1

CLASS_NAMES = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry_(including_sour)___Powdery_mildew", "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy",
    "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch", "Strawberry___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight",
    "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy",
]
NUM_CLASSES = len(CLASS_NAMES)

tf.random.set_seed(SEED)
np.random.seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("Clases:", NUM_CLASSES, "| Imagen objetivo:", IMAGE_TARGET_SIZE, "| Batch:", BATCH_SIZE)


In [ ]:
# 3. Cargar manifiestos y crear tf.data.Dataset (determinista, sin fuga entre splits)
import hashlib, csv, random
from collections import Counter, defaultdict

random.seed(SEED)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()

def leaf_name(nombre):
    stem = Path(nombre).stem
    if stem.endswith("_final_masked"):
        stem = stem[:-len("_final_masked")]
    partes = stem.split("___")
    if len(partes) >= 2:
        return f"{partes[0]}___{partes[1]}"
    return stem

# hash -> lista de (categoria, nombre)
hash_a_items = defaultdict(list)
total = 0
for clase_dir in sorted(data_dir.iterdir()):
    if not clase_dir.is_dir():
        continue
    categoria = clase_dir.name
    for img in sorted(clase_dir.iterdir()):
        if img.suffix.lower() in (".jpg", ".jpeg", ".png"):
            h = sha256_file(img)
            hash_a_items[h].append((categoria, img.name))
            total += 1

items_unicos = []
conflictos = []
for h, items in hash_a_items.items():
    cats = Counter(c for c, _ in items)
    categoria_elegida = cats.most_common(1)[0][0]
    if len(cats) > 1:
        conflictos.append((h, dict(cats), categoria_elegida))
    items_unicos.append((h, categoria_elegida, items[0][1]))

random.shuffle(items_unicos)
n = len(items_unicos)
n_train = max(1, round(n * TRAIN_SPLIT))
n_val = max(1, round(n * VAL_SPLIT))
n_test = n - n_train - n_val
if n_test < 1:
    n_train = max(1, n - 2)
    n_val = 1
    n_test = 1

grupos = {
    "train": items_unicos[:n_train],
    "val": items_unicos[n_train:n_train + n_val],
    "test": items_unicos[n_train + n_val:],
}

splits_rows = {"train": [], "val": [], "test": []}
for split, items in grupos.items():
    for h, categoria, _ in items:
        for categoria_arch, nombre_arch in hash_a_items[h]:
            splits_rows[split].append({
                "archivo": nombre_arch,
                "categoria": categoria,
                "split": split,
                "hash": h,
                "hoja": leaf_name(nombre_arch),
            })

# Verificación: ningún hash en más de un split
hash_a_split = {}
for split in ("train", "val", "test"):
    for r in splits_rows[split]:
        h = r["hash"]
        assert h not in hash_a_split or hash_a_split[h] == split, f"hash compartido: {h}"
        hash_a_split[h] = split

clases = sorted({c for items in hash_a_items.values() for c, _ in items})
assert set(clases) == set(CLASS_NAMES), "Clases del manifiesto no coinciden con CLASS_NAMES"

for split in ("train", "val", "test"):
    cats = {r["categoria"] for r in splits_rows[split]}
    assert cats == set(CLASS_NAMES), f"{split} no tiene las {NUM_CLASSES} clases"

print(f"Archivos totales: {total} | Únicas: {len(items_unicos)} | Clases: {len(clases)}")
print(f"train: {len(splits_rows['train'])} | val: {len(splits_rows['val'])} | test: {len(splits_rows['test'])}")
print(f"Conflictos de etiquetado documentados: {len(conflictos)}")


In [ ]:
# 4. tf.data desde manifiestos y pesos de clase (solo train)

def build_dataset(rows, shuffle=False):
    paths = [str(data_dir / r["categoria"] / r["archivo"]) for r in rows]
    labels = [CLASS_NAMES.index(r["categoria"]) for r in rows]
    ds = tf.data.Dataset.from_tensor_slices((paths, tf.constant(labels, dtype=tf.int32)))

    def load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMAGE_TARGET_SIZE, antialias=True)
        img = tf.cast(img, tf.float32)  # [0,255]
        return img, label

    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(min(len(paths), 10_000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = build_dataset(splits_rows["train"], shuffle=True)
val_ds = build_dataset(splits_rows["val"], shuffle=False)
test_ds = build_dataset(splits_rows["test"], shuffle=False)

# Pesos de clase SOLO con train
train_labels = np.concatenate([labels.numpy() for _, labels in train_ds], axis=0)
counts = Counter(train_labels.tolist())
class_weight = {c: len(train_labels) / (NUM_CLASSES * counts.get(c, 1)) for c in range(NUM_CLASSES)}

print("Train batches:", train_ds.cardinality().numpy(), "| Val:", val_ds.cardinality().numpy(), "| Test:", test_ds.cardinality().numpy())
print("Pesos clase (primeros 5):", {k: round(v, 3) for k, v in sorted(class_weight.items())[:5]})


In [ ]:
# 5. Modelo MobileNetV2 (preprocesamiento interno, una sola vez)

img_height, img_width = IMAGE_TARGET_SIZE
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(img_height, img_width, 3))
# Preprocesamiento oficial MobileNetV2: [0,255] -> [-1,1], aplicado una sola vez
x = tf.keras.layers.Lambda(
    lambda inputs: tf.keras.applications.mobilenet_v2.preprocess_input(inputs),
    name="mobilenetv2_preprocess",
)(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.2, name="classification_dropout")(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(x)
model = tf.keras.Model(inputs, outputs, name="mobilenetv2_baseline")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
model.summary()
print("base_trainable=False | fine_tuned=False (baseline)")


In [ ]:
# 6. Entrenamiento con class_weight (train) y early stopping

import mlflow
import mlflow.tensorflow
from codecarbon import EmissionsTracker

mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("leafthealthvision")

early_stop = tf.keras.callbacks.EarlyStopping(
    patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True
)

tracker = EmissionsTracker(save_to_file=True, output_dir="./results", output_file="emissions")

with mlflow.start_run(run_name=f"mobilenetv2_baseline_seed{SEED}"):
    mlflow.log_params({
        "model": "mobilenetv2",
        "variant": "baseline",
        "base_trainable": False,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": OPTIMIZER,
        "learning_rate": 0.001,
        "seed": SEED,
        "class_weight_mode": "train_only",
    })

    tracker.start()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weight,
        callbacks=[early_stop],
    )
    emisiones_kg = tracker.stop() or 0.0
    print(f"Huella de carbono: {emisiones_kg:.6f} kg CO2eq")


In [ ]:
# 7. Evaluación final bloqueada en test

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support,
)

y_true, y_pred = [], []
for images, labels in test_ds:
    logits = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(logits, axis=1).tolist())
y_true = np.array(y_true)
y_pred = np.array(y_pred)

labels_idx = list(range(NUM_CLASSES))
test_accuracy = float(accuracy_score(y_true, y_pred))
test_balanced = float(balanced_accuracy_score(y_true, y_pred))
test_macro_f1 = float(f1_score(y_true, y_pred, labels=labels_idx, average="macro", zero_division=0))
test_weighted_f1 = float(f1_score(y_true, y_pred, labels=labels_idx, average="weighted", zero_division=0))
p_macro, r_macro, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels_idx, average="macro", zero_division=0)
p_weighted, r_weighted, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels_idx, average="weighted", zero_division=0)

cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
report_text = classification_report(y_true, y_pred, labels=labels_idx, target_names=CLASS_NAMES, zero_division=0)

# Umbral de baja confianza SOLO con validación
conf_aciertos_val = []
for images, labels in val_ds:
    probs_val = tf.nn.softmax(model.predict(images, verbose=0)).numpy()
    pred_val = np.argmax(probs_val, axis=1)
    conf_val = np.max(probs_val, axis=1)
    correct_val = pred_val == labels.numpy()
    conf_aciertos_val.extend(conf_val[correct_val].tolist())
umbral_confianza = float(np.percentile(conf_aciertos_val, 20.0)) if conf_aciertos_val else 0.5

print("=== Métricas de test ===")
for k, v in {
    "test_accuracy": test_accuracy,
    "test_balanced_accuracy": test_balanced,
    "test_macro_f1": test_macro_f1,
    "test_weighted_f1": test_weighted_f1,
    "test_macro_precision": float(p_macro),
    "test_macro_recall": float(r_macro),
    "test_weighted_precision": float(p_weighted),
    "test_weighted_recall": float(r_weighted),
    "test_n_samples": int(len(y_true)),
    "umbral_confianza_validacion": umbral_confianza,
}.items():
    print(f"  {k}: {v}")
print("\n=== Reporte por clase ===")
print(report_text)


In [ ]:
# 8. Guardar artefactos (modelo, métricas, matriz, reporte, historial)

import json as _json
from datetime import datetime, timezone

run_dir = Path("./experiments/mobilenetv2/baseline/seed_42")
run_dir.mkdir(parents=True, exist_ok=True)

model.save(str(run_dir / "model.keras"))

config = {
    "variant": "baseline",
    "seed": SEED,
    "model": "mobilenetv2",
    "base_trainable": False,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "preprocess": "mobilenet_v2.preprocess_input ([0,255]->[-1,1])",
    "umbral_confianza_validacion": umbral_confianza,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
with (run_dir / "config_effective.json").open("w", encoding="utf-8") as f:
    _json.dump(config, f, indent=2, ensure_ascii=False)

with (run_dir / "metrics.json").open("w", encoding="utf-8") as f:
    _json.dump({
        "test_accuracy": test_accuracy,
        "test_balanced_accuracy": test_balanced,
        "test_macro_f1": test_macro_f1,
        "test_weighted_f1": test_weighted_f1,
        "test_macro_precision": float(p_macro),
        "test_macro_recall": float(r_macro),
        "test_weighted_precision": float(p_weighted),
        "test_weighted_recall": float(r_weighted),
        "test_n_samples": int(len(y_true)),
        "carbon_emissions_kg": float(emisiones_kg),
        "umbral_confianza_validacion": float(umbral_confianza),
    }, f, indent=2, ensure_ascii=False)

with (run_dir / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([""] + CLASS_NAMES)
    for i, row in enumerate(cm):
        w.writerow([CLASS_NAMES[i]] + [int(v) for v in row])

fig = plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Matriz de confusión (test)")
plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=90, fontsize=6)
plt.yticks(ticks, CLASS_NAMES, fontsize=6)
plt.ylabel("Clase real")
plt.xlabel("Clase predicha")
plt.tight_layout()
plt.savefig(run_dir / "confusion_matrix.png", bbox_inches="tight", dpi=150)
plt.close(fig)

with (run_dir / "classification_report.txt").open("w", encoding="utf-8") as f:
    f.write(report_text)

hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with (run_dir / "history.json").open("w", encoding="utf-8") as f:
    _json.dump(hist, f, indent=2, ensure_ascii=False)

mlflow.log_metrics({
    "test_accuracy": test_accuracy,
    "test_macro_f1": test_macro_f1,
    "carbon_emissions_kg": float(emisiones_kg),
})
mlflow.log_param("umbral_confianza_validacion", float(umbral_confianza))
mlflow.log_artifact(str(run_dir / "metrics.json"))
mlflow.log_artifact(str(run_dir / "classification_report.txt"))
mlflow.log_artifact(str(run_dir / "confusion_matrix.csv"))
mlflow.log_artifact(str(run_dir / "confusion_matrix.png"))
try:
    mlflow.tensorflow.log_model(model, name="model")
except Exception as exc:  # noqa: BLE001
    print(f"No se pudo registrar en MLflow: {exc}")

print(f"\nArtefactos guardados en: {run_dir}")


In [ ]:
# 9. Análisis de errores (al menos 10 casos erróneos, revisión humana)

casos_error = []
for fila in splits_rows["test"]:
    if len(casos_error) >= 10:
        break
    ruta = data_dir / fila["categoria"] / fila["archivo"]
    if not ruta.exists():
        continue
    img = tf.keras.utils.load_img(str(ruta), target_size=IMAGE_TARGET_SIZE)
    batch = np.expand_dims(np.asarray(img), 0).astype(np.float32)
    logits = model.predict(batch, verbose=0)
    probs = tf.nn.softmax(logits).numpy()[0]
    pred_idx = int(np.argmax(probs))
    confianza = float(probs[pred_idx])
    real_idx = CLASS_NAMES.index(fila["categoria"])
    if pred_idx != real_idx:
        casos_error.append({
            "clase_real": fila["categoria"],
            "clase_predicha": CLASS_NAMES[pred_idx],
            "confianza": confianza,
            "archivo": str(ruta),
            "tipo_error": "pendiente_revision_humana",
            "hipotesis_causa": "pendiente_revision_humana",
        })

import csv as _csv
with (run_dir / "casos_error_test.csv").open("w", newline="", encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=["clase_real", "clase_predicha", "confianza", "archivo", "tipo_error", "hipotesis_causa"])
    w.writeheader()
    w.writerows(casos_error)

print(f"Casos erróneos guardados: {len(casos_error)} en {run_dir / 'casos_error_test.csv'}")
for c in casos_error[:3]:
    print(c)


In [ ]:
# 10. Inferencia de demostración (Grad-CAM) — imagen de ejemplo, NO es PlantDoc

import csv, time
from datetime import datetime, timezone
from IPython.display import display
from PIL import Image

demo_url = "https://static.vecteezy.com/system/resources/previews/011/855/781/non_2x/green-leaf-of-apple-tree-photo.jpg"
demo_path = WORK / "test.jpg"
!curl -sL {demo_url} --output {demo_path}

image = tf.keras.utils.load_img(str(demo_path), target_size=IMAGE_TARGET_SIZE)
batch = np.expand_dims(np.asarray(image), 0).astype(np.float32)
logits = model.predict(batch, verbose=0)
probs = tf.nn.softmax(logits).numpy()[0]
idx = int(np.argmax(probs))
pred_class = CLASS_NAMES[idx]
conf = float(probs[idx])

print(f"Predicción: {pred_class} | Confianza: {conf:.2%} | Umbral val: {umbral_confianza:.2%}")
if conf < umbral_confianza:
    print("AVISO: baja confianza — resultado poco fiable.")

display(image)


In [ ]:
# 11. Mapa de calor Grad-CAM (clase predicha)

tracker = EmissionsTracker(save_to_file=False, log_level="ERROR")
tracker.start()
start = time.perf_counter()

def last_convolution(m):
    conv_layers = []
    def walk(current):
        for layer in current.layers:
            if isinstance(layer, tf.keras.Model):
                walk(layer)
            elif len(layer.output.shape) == 4:
                conv_layers.append(layer)
    walk(m)
    return conv_layers[-1]

layer = last_convolution(model)
activations = {}
original_call = layer.call

def capture(*args, **kwargs):
    output = original_call(*args, **kwargs)
    activations["feature_map"] = output
    return output

layer.call = capture
try:
    with tf.GradientTape() as tape:
        predictions = model(batch, training=False)
        probabilities = tf.nn.softmax(tf.cast(predictions, "float32"))[0]
        index = int(tf.argmax(probabilities))
        score = tf.cast(predictions[0, index], "float32")
        feature_map = activations["feature_map"]
finally:
    layer.call = original_call

gradients = tape.gradient(score, feature_map)
weights = tf.reduce_mean(tf.cast(gradients, "float32"), axis=(1, 2))
cam = tf.reduce_sum(tf.cast(feature_map, "float32") * weights[:, None, None, :], axis=-1)
cam = tf.nn.relu(cam)[0]
cam = cam / (tf.reduce_max(cam) + 1e-8)
heatmap = tf.image.resize(cam[..., None], IMAGE_TARGET_SIZE)[..., 0].numpy()

predicted_class = CLASS_NAMES[index]
confidence = float(probabilities[index].numpy())

latency_s = time.perf_counter() - start
emisiones_kg = tracker.stop() or 0.0

print(f"{predicted_class}: {confidence:.2%}")
print(f"Latencia: {latency_s:.3f} s | Emisiones: {emisiones_kg:.9f} kg CO2eq")

fig, (ax_original, ax_heatmap) = plt.subplots(1, 2, figsize=(10, 5))
ax_original.imshow(image)
ax_original.set_title("Original")
ax_heatmap.imshow(image)
ax_heatmap.imshow(heatmap, cmap="jet", alpha=0.5)
ax_heatmap.set_title(f"{predicted_class} ({confidence:.2%})")
for ax in (ax_original, ax_heatmap):
    ax.axis("off")
plt.tight_layout()
plt.show()
